# Week 4 lab — how long does an agent stay busy?

Week 3 used an exponential service time with a mean of six minutes. Week 1 counted calls entering the phone system in each 15-minute interval. This time, one observation is the number of seconds a human agent spends on a call. **How much work do long services create, why did their share increase, and when does that matter for staffing?**

Make your own copy from the repository root. Use the Python environment from Week 1.

```bash
mkdir -p students
cp notebooks/lecture4_parameter_estimation.ipynb students/lecture4_parameter_estimation_work.ipynb
```

This lab uses the Python standard library and two teaching extracts in `data/`. Start Jupyter from the repo root or open the local copy in VS Code. See the [data guide](../data/anonymousbank_data_guide.md) for how calls are recorded. The extracts contain aggregated counts and sums.

**Lab route (about 90 minutes).** Allow 20 minutes for fitting and checking, 30 for investigating long services, 30 for the queue comparisons, and 10 for the handoff.

The complete bank case will also need a model of who requests an agent, how long callers are willing to wait, and how the bank assigns staff. Today we work on **time spent with a human agent**.

## 1. Observe and screen the service records

Start with `AGENT` outcomes on Sunday–Thursday in November 1999. Keep calls whose **service started** from 09:00 to just before 11:00. Include their full durations even if they ended after 11:00. `ser_time` is the time between `ser_start` and `ser_exit`, in seconds. A caller who hangs up before service contributes no service duration to this analysis.

On comparable mornings, the share of recorded services under 10 seconds was 4.49% in January and 0.55% in November. The bank manager traced many of them to agents hanging up, as reported by [Brown et al. (2005), §4.1](https://faculty.wharton.upenn.edu/wp-content/uploads/2012/04/Brown-etal-CCdata.pdf).

For this lab, exclude the 28 November records **under 10 seconds** before fitting or checking a service model.

**Predict.** A service started at 10:55 and ended at 11:10. Which bin contains its full duration? How would cutting durations off at 11:00 change the share of long calls?

In [1]:
import csv
import json
import math
import random
import statistics
from datetime import datetime
from pathlib import Path

def data_file(name):
    here = Path.cwd()
    for folder in (here, *here.parents):
        candidate = folder / "data" / name
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"Cannot find data/{name}. Open the lab inside your course clone.")

with data_file("anonymousbank_nov1999_morning_service_summary.json").open(
    encoding="utf-8"
) as handle:
    summary = json.load(handle)

edges = summary["lower_edges_seconds"]
cutoff = 10
assert edges == sorted(set(edges)) and edges[:2] == [0, cutoff]

date_labels = {"training": "1–14 Nov.", "held_out": "15–30 Nov."}
for split, dates in date_labels.items():
    group = summary[split]
    counts = group["counts"]
    assert len(counts) == len(edges) and all(count >= 0 for count in counts)
    print(f"{dates}: {sum(counts):,} selected, {counts[0]} below 10 s, "
          f"{sum(counts[1:]):,} retained")

print("\nService durations, 1–14 November (one # represents about 15 records)")
upper_edges = [*edges[1:], None]
for lower, upper, count in zip(edges, upper_edges, summary["training"]["counts"]):
    label = f"[{lower}, {upper})" if upper is not None else f"{lower}+"
    print(f"{label:>13} s  {count:>4}  {'#' * round(count / 15)}")

1–14 Nov.: 2,544 selected, 14 below 10 s, 2,530 retained
15–30 Nov.: 2,528 selected, 14 below 10 s, 2,514 retained

Service durations, 1–14 November (one # represents about 15 records)
      [0, 10) s    14  #
     [10, 30) s    91  ######
     [30, 60) s   332  ######################
     [60, 90) s   435  #############################
    [90, 120) s   329  ######################
   [120, 180) s   483  ################################
   [180, 240) s   262  #################
   [240, 360) s   254  #################
   [360, 480) s   124  ########
   [480, 600) s    79  #####
   [600, 900) s    87  ######
  [900, 1200) s    19  #
 [1200, 1800) s    22  #
        1800+ s    13  #


The rows above are counts in intervals of unequal width. For example, `[120, 180)` includes 120 seconds and stops before 180. The bars show **number of records per interval**.

We fit using the regular working days from **1–14 November**. We save **15–30 November** for the check. The first set supplies the fitted parameters. The second lets us check how well the fitted models predict on days set aside from the fit.

## 2. Fit two candidates on 1–14 November

We fit an exponential and a lognormal distribution to the retained services from 1–14 November. Let $S$ denote service duration in seconds, and $S_i$ the duration of the $i$th retained service.

The extract contains the exact sums over the retained durations. If there are $n$ retained services, their mean is $m=\sum_i S_i/n$ and their empirical variance is $v=\sum_i S_i^2/n-m^2$. We use divisor $n$ to match the moments of the retained records.

- **Exponential:** choose its mean to be $m$. This also fixes its standard deviation to $m$.
- **Lognormal:** model $\log S$ as normal with mean $\mu$ and standard deviation $\sigma$. Match the observed mean and variance of $S$:

$$
\sigma^2=\log(1+v/m^2),
\qquad
\mu=\log m-\frac{\sigma^2}{2}.
$$

**Task 1.** Fill in `fit_lognormal`. Return `(mu, sigma)`. `math.log1p(z)` computes $\log(1+z)$ and `math.sqrt` takes the square root. The checks reconstruct the mean and variance for a small example.

In [ ]:
def retained_moments(group):
    n = sum(group["counts"][1:])
    # YOUR CODE: compute the mean and variance of the retained service durations.
    return mean_service, variance

def fit_lognormal(mean_duration, variance):
    # YOUR CODE: compute log-scale mu and sigma from the two empirical moments.
    raise NotImplementedError("Complete Task 1")

test_mu, test_sigma = fit_lognormal(100.0, 10_000.0)
assert math.isclose(math.exp(test_mu + test_sigma**2 / 2), 100.0)
assert math.isclose(100.0**2 * math.expm1(test_sigma**2), 10_000.0)

mean_service, variance = retained_moments(summary["training"])
mu, sigma = fit_lognormal(mean_service, variance)
print(f"Training mean {mean_service:.1f} s; SD {math.sqrt(variance):.1f} s.")
print(f"Both fitted means: {mean_service:.1f} s.")
print(f"Fitted exponential SD: {mean_service:.1f} s; lognormal SD: {math.sqrt(variance):.1f} s.")

## 3. Check predictions on 15–30 November

### Task 2 — What share of services last at least 10 minutes?

Add the counts from the bin starting at 600 seconds onward. Divide by the total number of retained services.

Complete `fraction_at_least`. Use `edges.index(threshold)` to find the first bin to include. The slice `counts[index:]` gives the counts at or above the threshold. Use `counts[1:]` for the total, since the first bin contains the services under 10 seconds that we screened out.

In [ ]:
def fraction_at_least(edges, counts, threshold):
    if threshold < edges[1]:
        raise ValueError("Use a threshold within the retained range")
    # YOUR CODE: use bin counts to compute the fraction among retained services.
    raise NotImplementedError("Complete Task 2")

# Three of five retained services last at least 30 seconds.
assert math.isclose(fraction_at_least([0, 10, 30, 60], [1, 2, 2, 1], 30), 0.60)


### Read the model comparison

The next cell uses your function for the **observed** percentages on 1–14 and 15–30 November. It also computes what each model predicts from its fitted distribution. Both models were fitted on 1–14 November, so use **15–30 November** to check their predictions.

1. Find the **10 min** row. For each model, say whether its prediction is above or below the observed share, and by how many **percentage points**. For example, a prediction of 3% against 5% observed is 2 percentage points below.
2. Which model is closer at **10 min**? Which is closer at **6 min**?
3. Compare the two observed percentages in the 10 min row. We will investigate that change in the next section.

In [ ]:
# For a standard normal, P(Z >= z) = erfc(z / sqrt(2)) / 2.
def exponential_tail(threshold):
    return math.exp(-threshold / mean_service)

def lognormal_tail(threshold):
    if threshold <= 0:
        return 1.0
    z = (math.log(threshold) - mu) / sigma
    return math.erfc(z / math.sqrt(2)) / 2

print("Share of services lasting at least the stated time")
print(f"{'At least':>9} {'1–14 Nov.':>11} {'15–30 Nov.':>11} "
      f"{'Exp. model':>12} {'Lognormal':>11}")
print(f"{'':>9} {'(fit)':>11} {'(check)':>11}")
for t in (120, 360, 600, 900):
    early = fraction_at_least(edges, summary["training"]["counts"], t)
    late = fraction_at_least(edges, summary["held_out"]["counts"], t)
    duration = f"{t // 60} min"
    print(f"{duration:>9} {early:11.1%} {late:11.1%} "
          f"{exponential_tail(t):12.1%} {lognormal_tail(t):11.1%}")

## 4. Investigate why the long services matter

### Task 3 — Count calls and count work

Imagine two calls take 2 minutes and 10 minutes. One out of two is a long call. That one call accounts for 10 of the 12 minutes of work.

Use the same idea on the bank records. For this comparison, a **long service** lasts **10 minutes or more**. The extract gives the total number of seconds of service and the seconds spent on long services. Complete `workload_share` to calculate the share of recorded service time from long calls, pooling the two service groups. For each set of dates, compare two quantities. Divide the number of long calls by the total number of retained calls. Then divide the service time from those same long calls by the total recorded service time.

**Explain.** Why could a model that underestimates the share of long calls also underestimate a substantial share of the work? Use the two percentages for 15–30 November. The calculation sums the full service durations of the selected calls.

In [ ]:
long_threshold = summary["long_service_seconds"]
assert long_threshold in edges
long_index = edges.index(long_threshold)
service_groups = {split: summary[split]["service_groups"] for split in ("training", "held_out")}
for split, groups in service_groups.items():
    assert sum(g["retained_count"] for g in groups.values()) == sum(summary[split]["counts"][1:])
    assert sum(g["long_count"] for g in groups.values()) == sum(summary[split]["counts"][long_index:])
    assert sum(g["service_seconds"] for g in groups.values()) == summary[split]["sum_seconds"]

def workload_share(groups):
    # YOUR CODE: sum the relevant seconds across both categories.
    raise NotImplementedError("Complete Task 3")

toy_work = {
    "A": {"service_seconds": 120, "long_seconds": 0},
    "B": {"service_seconds": 600, "long_seconds": 600},
}
assert math.isclose(workload_share(toy_work), 5 / 6)

print(f"{'Dates':<12} {'Share of calls':>15} {'Share of service time':>23}")
for split, groups in service_groups.items():
    count_share = fraction_at_least(edges, summary[split]["counts"], long_threshold)
    print(f"{date_labels[split]:<12} {count_share:15.1%} {workload_share(groups):23.1%}")

### Task 4 — Can the service mix explain the rise in long calls?

AnonymousBank records six service types. For this comparison, we pool them into two groups:

- **Regular banking:** standard banking (`PS`) and banking in English (`PE`).
- **Other services:** internet help (`IN`), stock exchange (`NE`), prospective customers (`NW`), and callbacks that entered the queue when the agent became busy (`TT`).

Read the two percentages for each row. **Share of calls** gives the size of the group relative to all retained calls for those dates. **Long within group** gives the share lasting 10 minutes or more among calls in that group.

Use the group sizes from **15–30 November** and keep each group's long-call rate at its **1–14 November** value. For example, with a later mix of 75% in group A and 25% in group B, and early long-call rates of 4% and 12%, the result is $0.75(0.04)+0.25(0.12)=0.06$.

Complete `weighted_long_share(mix_groups, rate_groups)`. Take the weights from the retained counts in `mix_groups` and the long-call rates from `rate_groups`. Multiply each group's weight by its rate, then add the products.

Compare the three percentages printed below. How much would the share of long calls have increased if only the mix had changed? How does that compare with the actual increase? Give both changes in **percentage points**.

In [ ]:
labels = {"regular_banking": "Regular banking", "other_services": "Other services"}
print(f"{'Dates':<12} {'Service group':<17} {'Calls':>7} {'Share of calls':>15} {'Long within group':>18}")
for split, groups in service_groups.items():
    total = sum(group["retained_count"] for group in groups.values())
    for category, group in groups.items():
        count = group["retained_count"]
        print(f"{date_labels[split]:<12} {labels[category]:<17} {count:7,} "
              f"{count / total:15.1%} {group['long_count'] / count:18.1%}")

def weighted_long_share(mix_groups, rate_groups):
    # YOUR CODE: calculate a weighted average of the category rates.
    raise NotImplementedError("Complete Task 4")

toy_mix = {"A": {"retained_count": 75}, "B": {"retained_count": 25}}
toy_rates = {
    "A": {"retained_count": 100, "long_count": 4},
    "B": {"retained_count": 100, "long_count": 12},
}
assert math.isclose(weighted_long_share(toy_mix, toy_rates), 0.06)

early = service_groups["training"]
later = service_groups["held_out"]
early_observed = weighted_long_share(early, early)
mix_only = weighted_long_share(later, early)
later_observed = weighted_long_share(later, later)
print(f"Observed on 1–14 November:   {early_observed:.2%}")
print(f"If only the mix had changed: {mix_only:.2%}")
print(f"Observed on 15–30 November:  {later_observed:.2%}")

## 5. Pair the simulations and compare staffing levels

We reuse the Week 3 arrival generator and queue rules. The arrival counts are entries to the automated phone menu. The simulation treats each entry as a request for an agent, even though some callers leave before asking for one. The simulated queue starts empty at 09:00. Callers wait until they are served, in order of arrival.

**Across service models:** generate one morning of arrivals. Draw a fresh uniform number between 0 and 1 for **every caller**. Reuse a caller's draw in all three service models. A draw of 0.95 selects the duration by which 95% of services finish under each model. The example after the setup shows three callers with different draws.

**Across staffing levels:** for a given service model, give both runs the exact same arrivals and service durations. Only the number of agents changes. As in Week 3, pairing helps reduce the random variation in the comparison. All service durations entering the simulator are in minutes.

In [ ]:
def minutes_from_midnight(clock):
    hour, minute = map(int, clock.split(":"))
    return 60 * hour + minute

with data_file("anonymousbank_nov1999_arrivals_15min.csv").open(newline="", encoding="utf-8") as handle:
    rows = [
        {
            "date": row["date"],
            "minute": minutes_from_midnight(row["time"]),
            "entries": int(row["entries"]),
        }
        for row in csv.DictReader(handle)
    ]

assert len(rows) == 30 * 96
assert sum(row["entries"] for row in rows) == 40_722

# In Israel, the regular working days represented here are Sunday–Thursday.
regular_dates = sorted(
    {
        row["date"]
        for row in rows
        if datetime.strptime(row["date"], "%Y-%m-%d").weekday()
        not in (4, 5)
    }
)
training_dates = regular_dates[:10]

morning_counts = [
    row["entries"]
    for row in rows
    if row["date"] in training_dates
    and 9 * 60 <= row["minute"] < 11 * 60
]
fitted_morning_count = statistics.mean(morning_counts)

daily_totals = [
    sum(row["entries"] for row in rows if row["date"] == day)
    for day in training_dates
]
daily_mean = statistics.mean(daily_totals)
daily_variance = statistics.variance(daily_totals)
alpha = (daily_variance - daily_mean) / daily_mean**2

In [ ]:
def simulate_queue(arrivals, services, number_of_agents):
    if not isinstance(number_of_agents, int) or number_of_agents <= 0:
        raise ValueError("number_of_agents must be a positive integer")
    if len(arrivals) != len(services):
        raise ValueError("arrivals and services must have equal length")
    if any(arrival < 0 for arrival in arrivals):
        raise ValueError("arrival times must be non-negative")
    if any(service <= 0 for service in services):
        raise ValueError("service times must be positive")
    if any(later < earlier for earlier, later in zip(arrivals, arrivals[1:])):
        raise ValueError("arrivals must be sorted")

    agent_free = [0.0] * number_of_agents
    records = []
    for customer, (arrival, service) in enumerate(
        zip(arrivals, services), start=1
    ):
        agent_index = min(
            range(number_of_agents),
            key=lambda index: (agent_free[index], index),
        )
        start = max(arrival, agent_free[agent_index])
        wait = start - arrival
        departure = start + service
        agent_free[agent_index] = departure
        records.append(
            {
                "customer": customer,
                "arrival": arrival,
                "wait": wait,
                "start": start,
                "service": service,
                "departure": departure,
                "agent": agent_index + 1,
            }
        )
    return records

def generate_parametric_morning(rng, horizon=120.0):
    multiplier = rng.gammavariate(1 / alpha, alpha)
    rate_per_minute = multiplier * fitted_morning_count / 15
    arrivals = []
    clock = 0.0
    while True:
        clock += rng.expovariate(rate_per_minute)
        if clock >= horizon:
            break
        arrivals.append(clock)
    return arrivals, multiplier

def mean_wait(arrivals, services, number_of_agents):
    records = simulate_queue(arrivals, services, number_of_agents)
    waits = [record["wait"] for record in records]
    return statistics.mean(waits) if waits else 0.0

In [ ]:
normal = statistics.NormalDist()
scenarios = ("Week 3 exponential", "Fitted exponential", "Fitted lognormal")

def service_scenarios(uniforms):
    return {
        "Week 3 exponential": [-6 * math.log1p(-u) for u in uniforms],
        "Fitted exponential": [
            -mean_service * math.log1p(-u) / 60 for u in uniforms
        ],
        "Fitted lognormal": [
            math.exp(mu + sigma * normal.inv_cdf(u)) / 60 for u in uniforms
        ],
    }

def run_experiment(repetitions=200, arrival_seed=20260301, service_seed=20260302,
                   *, agents=(16, 17)):
    fewer_agents, more_agents = agents
    arrival_rng = random.Random(arrival_seed)
    service_rng = random.Random(service_seed)
    results = {label: [] for label in scenarios}
    for _ in range(repetitions):
        arrivals, _ = generate_parametric_morning(arrival_rng)
        # Keep u above zero because the normal quantile requires 0 < u < 1.
        uniforms = [max(service_rng.random(), 2**-53) for _ in arrivals]
        for label, services in service_scenarios(uniforms).items():
            wait_fewer = mean_wait(arrivals, services, fewer_agents)
            wait_more = mean_wait(arrivals, services, more_agents)
            results[label].append((wait_fewer, wait_more, wait_fewer - wait_more))
    return results

def summarize_effect(values):
    average = statistics.mean(values)
    half_width = 1.96 * statistics.stdev(values) / math.sqrt(len(values))
    return average, (average - half_width, average + half_width)

# Illustrative draws for three callers. run_experiment draws for every arrival.
example_draws = [0.25, 0.60, 0.95]
example_services = service_scenarios(example_draws)
print("Example service durations (minutes)")
print(f"{'Caller':>6} {'Draw':>6} {'Week 3 exp.':>12} "
      f"{'Fitted exp.':>12} {'Fitted lognormal':>17}")
for caller, draw in enumerate(example_draws, start=1):
    durations = [example_services[model][caller - 1] for model in scenarios]
    old_exp, fitted_exp, fitted_log = durations
    print(f"{caller:6d} {draw:6.2f} {old_exp:12.2f} {fitted_exp:12.2f} {fitted_log:17.2f}")

### Task 5a — Keep staffing at 16 and 17 agents

For every caller in each of 200 mornings, generate service durations as above. Under each service model, run the morning with 16 and 17 agents. Compute the mean wait at each staffing level **for that morning**. The table averages these 200 morning means. The **reduction** is how much the mean wait falls when adding the agent. The Monte Carlo interval describes precision with the inputs fixed.

Which change has a larger effect on the reduction at 16 versus 17: updating the six-minute assumption, or replacing the fitted exponential with the fitted lognormal? Convert the two fitted reductions from minutes to seconds before judging their size.

In [ ]:
results = run_experiment()
print("Average of 200 morning means (minutes)")
print(f"{'Service model':<22} {'Wait at 16':>10} {'Wait at 17':>10} "
      f"{'Reduction':>10} {'95% MC interval':>20}")
for label, rows in results.items():
    wait_16, wait_17, reductions = zip(*rows)
    average, (lower, upper) = summarize_effect(reductions)
    print(f"{label:<22} {statistics.mean(wait_16):10.3f} {statistics.mean(wait_17):10.3f} "
          f"{average:10.3f} [{lower:.3f}, {upper:.3f}]")

### Task 5b — Compare mean waiting times closer to capacity

Use the same reference demand as in Week 3, 39.85 calls per 15 minutes. Multiply the arrival rate in calls per minute by the fitted mean service duration in minutes. The result is about 9.11 agent minutes of work arriving per minute. At that demand, 16 agents have about 6.9 agent minutes of spare capacity per minute. Ten agents have about 0.9. We test 10 versus 11 agents to see how much one extra agent helps when spare capacity is smaller.

Use the same 200 simulated mornings and the same fitted service models at both staffing pairs. The table shows the **mean waiting time in seconds** at 16, 17, 10, and 11 agents. For each model, subtract the mean wait with one more agent from the mean wait with fewer agents. Which of the two staffing changes saves more waiting time? Explain the result in terms of spare capacity.

In [ ]:
offered_work = (fitted_morning_count / 15) * (mean_service / 60)
print(f"Average-demand workload: {offered_work:.2f} agent minutes per minute")

lower_staffing = run_experiment(agents=(10, 11))
print("\nMean waiting time in seconds, averaged over 200 mornings")
print(f"{'Agents':>6} {'Fitted exponential':>20} {'Fitted lognormal':>18}")
for staffing, trial in (((16, 17), results), ((10, 11), lower_staffing)):
    for column, number_of_agents in enumerate(staffing):
        exponential_wait = statistics.mean(row[column] for row in trial["Fitted exponential"])
        lognormal_wait = statistics.mean(row[column] for row in trial["Fitted lognormal"])
        print(f"{number_of_agents:6d} {60 * exponential_wait:20.2f} {60 * lognormal_wait:18.2f}")

### Task 5c — Does the estimated reduction depend on the service model?

Task 5b compares staffing levels within a model. Now check whether the **reduction in mean wait from adding one agent** is larger under one service model.

In each simulation result, a row contains the mean wait with fewer agents, the mean wait with more agents, and their reduction, all in minutes. Use `zip` to match rows from the two fitted models, since each row represents the same simulated morning. For each matched morning, subtract the **exponential model's reduction** from the **lognormal model's reduction**. Pass the resulting list to `summarize_effect`. Print the average and its approximate 95% Monte Carlo interval in seconds by multiplying by 60.

For each staffing pair, does the interval give clear evidence that the fitted service model changes the reduction?

In [ ]:
print("Reduction in mean wait: lognormal model minus exponential model")
for label, trial in (("16 to 17", results), ("10 to 11", lower_staffing)):
    exponential_rows = trial["Fitted exponential"]
    lognormal_rows = trial["Fitted lognormal"]
    # YOUR CODE: form the paired differences and print their estimate and interval.
    raise NotImplementedError("Complete Task 5c")

## 6. Decide what to carry forward

Write a short handoff for Week 5:

1. State what the data say about the six-minute assumption for this cohort.
2. Explain how much of the increase in long services the two-group mix explains, and why the remaining gap matters for workload.
3. Compare the reduction in mean waiting time at the two staffing pairs. State what the paired intervals say about the effect of changing the fitted service model.
4. Explain one way to check whether the **queue** behaves like the bank. Name one measurement you need and where to find it in the [data guide](../data/anonymousbank_data_guide.md).

If you use an LLM, ask it to predict what changes if you switch the order of the mix and rate swaps. Explain its prediction in your own words and check it by running `weighted_long_share(early, later)`. Keep individual records and student work local.